# Forsteinrichtungsoperate – Ecological Information Extraction with Gemini 3.5 Flash (proof of concept)

Extracts the most important **ecological information** from the Markdown transcriptions
(`*.md`, produced by the Gemini-3-Flash transcription pipeline) of 19th/20th-century Bavarian forest
management plans (Forstamt St. Oswald / Klingenbrunn, Bavarian Forest).

Two source types are handled with two different prompts + JSON schemas:

| source type | input | output |
|---|---|---|
| **text pages** (Erörternde Darstellung, Protokolle, Schreiben, gedruckter Text) | prose md | a flat list of *findings* (category, entity, value, unit, date, location, verbatim quote) + page-level species list and locations |
| **tables** (Periodentabelle, Wirtschaftsplan, Kulturplan, Bestandsauszählungen, …) | md with `<table>` HTML | a list of *stand records* keyed by Distrikt / Abteilung / Unterabteilung with site, stand, damage, management, culture and non-timber-use sub-objects |

The IE is applied to the **md files only**, never to the scans.

**Test set:** `Forsteinrichtung_IE_testset.zip` (20 text pages + 20 table pages, chosen for ecological
richness and format diversity, see `manifest.csv`).

**Workflow:** 1 Setup → 2 Data → 3 Schema & prompts → 4 Dry run on 2 pages → 5 Batch run (cached, resumable)
→ 6 Flatten to tables, quote verification, review viewer → 7 Notes for scaling to the full corpus.

## 1 · Setup

In [ ]:
!pip install -q -U google-genai pandas tqdm

import os, re, json, time, html, zipfile, shutil, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from tqdm.auto import tqdm
from google import genai
from google.genai import types

IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    pass
print("Running in Colab:", IN_COLAB)

In [ ]:
# ---- API key: Colab secret "LST_Gemini" (as in the transcription pipeline) or GEMINI_API_KEY ----
GEMINI_API_KEY = ""
if IN_COLAB:
    from google.colab import userdata
    for secret in ("LST_Gemini", "GEMINI_API_KEY", "GOOGLE_API_KEY"):
        try:
            GEMINI_API_KEY = userdata.get(secret) or ""
            if GEMINI_API_KEY:
                print(f"API key loaded from Colab secret '{secret}'"); break
        except Exception:
            pass
if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY") or ""
if not GEMINI_API_KEY:
    import getpass
    GEMINI_API_KEY = getpass.getpass("Gemini API key: ")

client = genai.Client(api_key=GEMINI_API_KEY)

# ---- Model ----
MODEL_ID = "gemini-3.5-flash"          # GA model id; replaces gemini-3-flash-preview
THINKING_LEVEL = "medium"              # "low" | "medium" (default) | "high"
MAX_OUTPUT_TOKENS = 32768              # thinking tokens count against the output budget
WORKERS = 4                            # parallel requests

# prices in USD per 1M tokens – update from https://ai.google.dev/gemini-api/docs/pricing
PRICE_INPUT_PER_M  = 0.50
PRICE_OUTPUT_PER_M = 3.00

# sanity check: is the model visible to this key?
try:
    names = [m.name for m in client.models.list()]
    hits = [n for n in names if "3.5-flash" in n]
    print("gemini 3.5 flash models visible:", hits or "NONE – check MODEL_ID / API key")
except Exception as e:
    print("model listing failed (not fatal):", e)

## 2 · Data

Three ways to provide the test set (all end in `DATA_DIR` containing `manifest.csv`, `text/`, `tables/`):

* **A – upload** `Forsteinrichtung_IE_testset.zip` from your computer (default in Colab)
* **B – Google Drive**: set `DRIVE_ZIP` or `DRIVE_DIR` to the location on Drive
* **C – local**: running the notebook on your machine, point `LOCAL_DIR` to the unzipped folder

In [ ]:
DRIVE_ZIP = ""      # e.g. "/content/drive/MyDrive/Forsteinrichtung/Forsteinrichtung_IE_testset.zip"
DRIVE_DIR = ""      # e.g. "/content/drive/MyDrive/Forsteinrichtung/Forsteinrichtung_IE_testset"
LOCAL_DIR = "/home/tobiasperschl/Downloads/Forsteinrichtung_IE_testset"

WORK = Path("/content/ie_work") if IN_COLAB else Path.cwd() / "ie_work"
WORK.mkdir(parents=True, exist_ok=True)
OUT_DIR = WORK / "output"; OUT_DIR.mkdir(exist_ok=True)
RAW_DIR = OUT_DIR / "raw"; RAW_DIR.mkdir(exist_ok=True)     # one JSON per page (cache / resume)

def _unzip(zip_path, target):
    target = Path(target)
    if target.exists(): shutil.rmtree(target)
    target.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as z: z.extractall(target)
    # zip contains a top-level folder
    inner = [p for p in target.iterdir() if p.is_dir() and (p / "manifest.csv").exists()]
    return inner[0] if inner else target

DATA_DIR = None
if IN_COLAB and (DRIVE_ZIP or DRIVE_DIR):
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = _unzip(DRIVE_ZIP, WORK / "testset") if DRIVE_ZIP else Path(DRIVE_DIR)
elif IN_COLAB:
    from google.colab import files
    print("Upload Forsteinrichtung_IE_testset.zip …")
    up = files.upload()
    zip_name = next(iter(up))
    DATA_DIR = _unzip(zip_name, WORK / "testset")
else:
    DATA_DIR = Path(LOCAL_DIR)

manifest = pd.read_csv(DATA_DIR / "manifest.csv")
print("DATA_DIR:", DATA_DIR)
print(manifest.groupby("source_type").size())
manifest[["seq", "source_type", "subtype", "page_id", "chars"]]

### 2.1 · Pre-processing of the md files

* tolerant YAML-frontmatter parser (page_id contains `[`, `,`)
* marginalia blockquotes that degenerated into repetition loops are truncated
* line-end hyphenation (`-` and Kurrent `=`) is joined when the next line starts lowercase, so that quotes and
  entity strings are contiguous. The **cleaned text is what the model sees, and what quotes are verified against**.
* HTML tables: indentation collapsed to save tokens; structure untouched

In [ ]:
FM_RE = re.compile(r"^---\s*\n(.*?)\n---\s*\n?", re.S)

def parse_md(path):
    raw = Path(path).read_text(encoding="utf-8")
    meta, body = {}, raw
    m = FM_RE.match(raw)
    if m:
        for line in m.group(1).splitlines():
            if ":" in line:
                k, v = line.split(":", 1)
                meta[k.strip()] = v.strip().strip('"')
        body = raw[m.end():]
    return meta, body

def truncate_marginalia_loops(body, max_lines=25):
    out, run = [], []
    def flush():
        nonlocal run
        if run:
            if len(run) > max_lines:
                out.extend(run[:max_lines]); out.append(f"> *[marginalia truncated: {len(run) - max_lines} further lines]*")
            else:
                out.extend(run)
            run = []
    for line in body.splitlines():
        if line.startswith(">"):
            run.append(line)
        else:
            flush(); out.append(line)
    flush()
    return "\n".join(out)

HYPHEN_RE = re.compile(r"(\w)[-=]\n(?=[a-zäöüß])")
def dehyphenate(text):
    # 'Wind=\nrißflächen' -> 'Windrißflächen'; keeps 'Bau- u. Nutzholz' (next line not lowercase-continued)
    return HYPHEN_RE.sub(r"\1", text)

def compact_html(text):
    def _c(m):
        block = m.group(0)
        block = re.sub(r"\n\s+", "\n", block)                  # strip indentation
        block = re.sub(r"\n(?=</?(?:td|th|tr|thead|tbody)\b)", "", block)  # cell tags onto one line
        return block
    return re.sub(r"```html.*?```", _c, text, flags=re.S)

def prepare_page(row):
    meta, body = parse_md(DATA_DIR / row["file"])
    body = truncate_marginalia_loops(body)
    if row["source_type"] == "text":
        body = dehyphenate(body)
    else:
        body = compact_html(body)
    return {
        "seq": row["seq"], "source_type": row["source_type"], "subtype": row["subtype"],
        "page_id": meta.get("page_id", row["page_id"]), "file": row["file"],
        "meta": meta, "text": body.strip(), "chars": len(body),
    }

pages = [prepare_page(r) for _, r in manifest.iterrows()]
pd.DataFrame([{k: p[k] for k in ("seq", "subtype", "page_id", "chars")} for p in pages])

In [ ]:
# look at one prepared text page and one prepared table page
print(pages[0]["text"][:1500]); print("\n" + "=" * 100 + "\n"); print(pages[20]["text"][:2500])

## 3 · Extraction schemas and prompts

Two JSON schemas, passed to Gemini as `response_json_schema` (structured output). Every extracted item
carries a **verbatim quote** so the extraction can be verified against the transcription and later be linked
back to the scan via the region JSON of the pipeline.

In [ ]:
CATEGORIES = [
    "tree_species",          # Holzart, Anteil, Rolle (herrschend/beigemischt), Mischung
    "stand_structure",       # Bestandesform, Alter, Altersklasse, Umtrieb, Schluss/Bestockung, Urwald, Plenterwald
    "site_conditions",       # Lage, Höhe ü. M., Exposition, Hangneigung, Boden, Grundgestein, Feuchtigkeit, Auen/Filze
    "ground_vegetation",     # Bodenflora: Moos, Heidelbeere, Himbeere, Senecio, Sphagnum, Gras, Verunkrautung
    "climate_weather",       # Schnee, Spätfrost, Vegetationszeit, Böhmwind, Witterung
    "damage_event",          # Windwurf/Windbruch, Borkenkäfer, Schneebruch, Duftbruch, Wildverbiss, Rotfäule, Brand, Schälschaden
    "regeneration",          # Naturverjüngung, Anflug, Vorwuchs, Saat, Pflanzung, Vorbau, Erfolg/Misserfolg
    "silvicultural_measure", # Kahlschlag, Plenter-, Femel-, Saumschlag, Durchforstung, Läuterung, Räumung, Angriffshieb
    "non_timber_use",        # Weide/Hut, Streunutzung, Harz, Torf, Ameiseneier, Glashütten-Holz, Kohlen
    "wildlife",              # Reh, Rotwild, Insekten, Vögel
    "land_use_hydrology",    # Entwässerung, Wiesen, Nichtholzboden, Wege, Triftbau, Quellen, Bäche
    "quantity",              # Fläche, Vorrat, Zuwachs, Ertrag with unit (ha, Tagwerk, fm, Klafter, Ster)
    "other_ecological",
]

CONF = {"type": "string", "enum": ["high", "medium", "low"]}
S = lambda d="": {"type": "string", "description": d}
N = lambda d="": {"type": "number", "description": d}
ARR = lambda item: {"type": "array", "items": item}

TEXT_SCHEMA = {
    "type": "object",
    "properties": {
        "page_id": S(),
        "document_type": {"type": "string", "enum": [
            "Eroerternde_Darstellung", "Grundlagenprotokoll", "Wirtschaftsregeln", "Protokoll", "Schreiben",
            "Bericht", "Inhaltsverzeichnis", "Tabelle", "other"]},
        "summary_en": S("1-2 sentences: what the page is about (English)"),
        "forest_offices": ARR(S("Forstamt / Revier names mentioned, e.g. Klingenbrunn, St. Oswald, Riedlhütte")),
        "dates_mentioned": ARR(S("ISO-like dates or years/periods as written, e.g. 1870-10-26, 1873/74, 1878/90")),
        "locations": ARR({"type": "object", "properties": {
            "district_no": S("Distrikt number (Roman) if given"), "district_name": S(),
            "compartment_no": S("Abteilung No"), "subcompartment": S("Unterabteilung lit."),
            "toponym": S("place / stand name as written"),
            "area_value": N(), "area_unit": S("ha | Tagwerk | Dezimale | other"),
            "quote": S("verbatim snippet")}, "required": ["toponym", "quote"]}),
        "tree_species": ARR({"type": "object", "properties": {
            "name_de": S("normalised German name: Fichte, Tanne, Buche, Kiefer, Lärche, Ahorn, Esche, Birke, Erle, Ulme, Eiche, Zirbe, Douglasie, Strobe, Spirke, Latsche, Vogelbeere, Aspe …"),
            "name_latin": S(), "as_written": S("form in the text, e.g. Fi, Ta, Bu, Rothbuche"),
            "role": {"type": "string", "enum": ["dominant", "admixed", "understorey", "single_trees", "exotic_trial", "absent_or_declining", "unspecified"]},
            "share_value": N("0-1 fraction if a share is given (8/10 -> 0.8; 50% -> 0.5; WZ 0,7 -> 0.7)"),
            "share_basis": S("area | mass | WZ (Wirtschaftsziel) | stem_count | unspecified"),
            "quote": S()}, "required": ["name_de", "role", "quote"]}),
        "findings": ARR({"type": "object", "properties": {
            "category": {"type": "string", "enum": CATEGORIES},
            "subtype": S("free short label, e.g. windthrow, bark_beetle, elevation_limit, snow_cover, grazing_right, Femelschlag"),
            "entity": S("normalised German headword, e.g. Windwurf, Borkenkäfer, Fichte, Streunutzung, Gneis"),
            "entity_en": S("English gloss"),
            "species": ARR(S("normalised species names involved, if any")),
            "value": S("quantity / qualitative value as written, e.g. '1180 m', '8/10', 'Anfang Oktober bis Ende Mai'"),
            "unit": S(), "date": S("date/year/period the statement refers to, if any"),
            "location": S("Distrikt/Abteilung/toponym/Lage the statement refers to, if any"),
            "status": S("planned | executed | prohibited | recommended | observed | historical | ended – if applicable"),
            "quote": S("VERBATIM snippet from the text, max ~40 words, copied exactly"),
            "confidence": CONF,
            "note": S("uncertainty, HTR problems, interpretation")},
            "required": ["category", "entity", "quote", "confidence"]}),
        "transcription_quality": {"type": "string", "enum": ["good", "mixed", "poor"]},
        "continues_previous_page": {"type": "boolean"},
        "ends_mid_sentence": {"type": "boolean"},
    },
    "required": ["page_id", "document_type", "summary_en", "findings", "tree_species", "locations", "transcription_quality"],
}

SPECIES_ITEM = {"type": "object", "properties": {
    "name_de": S("normalised: Fichte, Tanne, Buche, …"), "as_written": S(),
    "share_value": N("0-1 fraction if given"), "count": N("stem count if given"),
    "volume_value": N(), "volume_unit": S(),
    "role": S("dominant | admixed | understorey | single_trees | unspecified"),
    "note": S()}, "required": ["name_de"]}

RECORD_SCHEMA = {"type": "object", "properties": {
    "record_kind": {"type": "string", "enum": ["stand", "sample_plot", "summary_row", "carry_over", "forest_office_total", "year_row", "other"]},
    "district_no": S("Distrikt No, Roman numeral as written"), "district_name": S(),
    "compartment_no": S("Abteilung No"), "subcompartment": S("Unterabteilung lit."),
    "stand_name": S("local name if given"), "forest_office": S("Forstamt / Revier if given for this row"),
    "year_or_period": S("year, period label (I. Periode 18--) or year range this row refers to"),
    "area_value": N("area as one decimal number (Hektar+Ar -> 18.60; Tagw+Dez -> 12.35)"),
    "area_unit": S("ha | Tagwerk | other"), "area_raw": S("cells as written"),
    "site": {"type": "object", "properties": {
        "elevation_min_m": N(), "elevation_max_m": N(), "exposition": S(), "slope": S(),
        "bedrock": S("Grundgestein"), "soil": S("Bodenart, Gründigkeit, Steinigkeit"),
        "moisture": S("frisch | trocken | nass | Moor | …"), "ground_vegetation": ARR(S()),
        "frost_exposure": S("Frostlage / ungeschützt / geschützt"), "site_class": S("Bonitäts-/Standortsklasse")}},
    "stand": {"type": "object", "properties": {
        "species": ARR(SPECIES_ITEM),
        "age_min": N(), "age_max": N(), "age_mean": N(), "age_raw": S(), "age_class": S(),
        "stocking_min": N("Bestockungsgrad 0-1"), "stocking_max": N(),
        "structure": S("Schluss, Stammklassen, Stockwerke, Urwald, Plenter …"),
        "health": S("gutwüchsig | rückgängig | …"), "regeneration": S("Verjüngung: art, erwartet/vorhanden"),
        "volume_per_unit": N(), "volume_total": N(), "volume_unit": S("Fmtr | fm | Ster | Klafter …"),
        "increment_value": N(), "increment_unit": S(),
        "stem_count_main": N(), "stem_count_secondary": N(), "stem_count_unit": S("pro ha | pro Tagwerk | absolute")}},
    "damage": ARR({"type": "object", "properties": {
        "type": S("Windbruch | Windwurf | Borkenkäfer | Schneebruch | Duftbruch | Schälschaden | Wildverbiss | Frost | Rotfäule | Weide | other"),
        "extent": S(), "year": S(), "quote": S()}, "required": ["type", "quote"]}),
    "management": {"type": "object", "properties": {
        "planned_cut_type": S("Saumschlag | Femelschlag | Kahlschlag | Plenterhieb | Angriffshieb | Durchforstung | Nachhieb | Abtrieb | Läuterung …"),
        "period": S(), "cut_area_value": N(), "cut_area_unit": S(), "cut_yield_value": N(), "cut_yield_unit": S(),
        "transition_code": S("Uebergangsbestimmungen Gattung/Jahr, e.g. P 70, D 160, N 280"),
        "notes": S("Bemerkungen über das Betriebsverfahren, verbatim-ish"),
        "history": ARR({"type": "object", "properties": {
            "year": S(), "cut_type": S(), "area_value": N(), "timber_value": N(), "fuelwood_value": N(),
            "total_value": N(), "unit": S()}, "required": ["year"]})}},
    "culture": ARR({"type": "object", "properties": {
        "type": S("Pflanzung | Saat | Lückenpflanzung | Aufforstung | Entwässerung | Schlagpflege | Unkrautreinigung | other"),
        "species": S(), "quantity": N(), "unit": S("Stück | kg | Pfd | Ruthen | Metzen …"),
        "area_value": N(), "area_unit": S(), "year": S(), "cost_value": N(), "cost_unit": S(), "raw": S()},
        "required": ["type"]}),
    "non_timber_use": ARR({"type": "object", "properties": {
        "type": S("Streunutzung | Weide | Torf | Harz | Mast | Borke | Erden/Steine | other"),
        "species_group": S("Nadel | Laub | Moos | Farn …"), "area_value": N(), "area_unit": S(),
        "quantity": N(), "unit": S("Fuhren | Karren | Ster | Stück Vieh …"), "year": S(), "raw": S()},
        "required": ["type"]}),
    "description_raw": S("verbatim content of the free-text description cell(s) (Beschreibung a-e, Bemerkungen, Boden/Lage/Holzbestand), joined"),
    "red_annotations": ARR(S("content of <span class='red'> cells for this row")),
    "cross_references": ARR(S("e.g. 'wie bei IV. 8. b', 'Seite 17'")),
    "uncertain": {"type": "boolean", "description": "row contains [?], [illegible] or ambiguous cell alignment"},
}, "required": ["record_kind", "uncertain"]}

TABLE_SCHEMA = {
    "type": "object",
    "properties": {
        "page_id": S(),
        "table_type": S("recognised form, e.g. Periodentabelle, Wirtschaftsplan, Kulturplan, Bestandsauszählung, Probeflächenaufnahme, Altersklassentabelle, Streunutzungsplan, Wirtschaftsbuch, Betriebsplan"),
        "forest_office": S("Forstamt / Revier from header lines"),
        "operating_class": S("Betriebsklasse / Umtrieb from header, e.g. 'Hochwald in 280 jhr. Umtriebe'"),
        "column_headers": ARR(S("flattened header path per data column, in order")),
        "period_labels": ARR(S()),
        "records": ARR(RECORD_SCHEMA),
        "marginalia": ARR(S()),
        "parsing_notes": S("column drift, two side-by-side blocks, continuation from previous page, etc."),
        "transcription_quality": {"type": "string", "enum": ["good", "mixed", "poor"]},
    },
    "required": ["page_id", "table_type", "records", "transcription_quality"],
}
print("schemas defined; text findings categories:", len(CATEGORIES))

In [ ]:
GLOSSARY = '''
GLOSSARY (local usage in the Bavarian Forest Forsteinrichtungsoperate, 1830s-1920s):
- Distrikt (Roman numeral + name, e.g. "VIII Sagwasserhang"), Abteilung (number), Unterabteilung (lit. a, b, c) = stand address.
- Holzarten & abbreviations: Fi/F = Fichte (Picea abies), Ta/T = Tanne (Abies alba), Bu/Bü/B/Rothbuche = Buche (Fagus sylvatica),
  Ki/Föhre/Kiefer (Pinus sylvestris), Lä/Lär/Lerche = Lärche (Larix decidua), Ah = Ahorn (Acer pseudoplatanus), Esch = Esche,
  Bi/Be = Birke, Er = Erle, Ul/Elbe = Ulme, Ei = Eiche, Zirbe (Pinus cembra), Spirke/Latsche (Pinus mugo), Vogelbeere, Aspe,
  Douglasie, Strobe (Pinus strobus), Sitkafichte, Höhenkiefer.
- Mixture notation: "8/10 Fi u. Ta, 2/10 Bu" = fractions of area; "50 % der Masse"; "0,7 Fi 0,2 Ta 0,1 Bu" = shares (Bestockung);
  "WZ 0,7 Fi 0,3 BuTa" = Wirtschaftsziel (target mixture). Bestockung(sgrad)/Schluß 0,4-1,0 = stocking density.
- Auen = waterlogged spruce flats on plateaus (NOT riparian); Filze = raised bogs; Hochwald = the high-elevation spruce zone
  (a local Betriebsklasse above ~1150 m); Urwald = primeval/unmanaged old forest; Plenter/Plänterwald = selection forest.
- Hiebsarten: Kahlschlag/Kahlhieb, Plenterhieb, Femelschlag, Saumschlag, Saumfemelschlag, Dunkelschlag, Sammelschlag, Angriffshieb,
  Nachhieb, Räumung(shieb), Durchforstung, Läuterung, Vorbereitungshieb. Übergangsbestimmungen codes: N = Nachhieb, P = Pflanzung, D = Durchforstung.
- Schäden: Windwurf/Windbruch/Windriß (storm), Borkenkäfer/Käferfraß (bark beetle), Schneebruch/Schneedruck (snow), Duftbruch (rime/ice),
  Wildverbiß (game browsing), Schälschaden (bark stripping), Rotfäule (root rot), Frost/Spätfrost, Verunkrautung (weed competition).
- Nebennutzungen: Weide/Hut/Tagweide (grazing), Streunutzung/Laub-, Nadel-, Moosstreu (litter raking), Harz (resin), Torf (peat),
  Ameiseneier, Glashütten/Glasfabrik (wood for glassworks), Kohlen (charcoal), Mast, Lohe/Borke.
- Units: ha + Ar (Hektar|Ar columns -> 18.60 ha), Tagwerk (Tgw) + Dezimale (1 Tgw = 0.3407 ha), Fmtr/fm (Festmeter = m³),
  Ster, Klafter (Klftr, Str), Kubikfuß, Ruthen, Pfd (pound), Metzen, fl./kr. (Gulden/Kreuzer), M/Pf (Mark/Pfennig).
- HTR markers: [?] uncertain reading, [illegible], [...] gap, "..." lost line start; <span class="red"> = later red-ink correction.
'''

TEXT_PROMPT = '''You are an expert in historical forestry (Forsteinrichtung) and ecological history, reading Gemini HTR transcriptions
of 19th/early-20th-century German (Kurrent) forest-management documents from the Bavarian Forest (Forstamt St. Oswald, Klingenbrunn,
Spiegelau, Riedlhütte, Schönau/Waldhäuser).

TASK: Extract ALL ecologically relevant information from the page below into the JSON structure requested.
Focus on: tree species and mixture shares, stand structure and age, site conditions (elevation, exposure, slope, soil, bedrock,
moisture, Auen/Filze), ground vegetation, climate/weather, damage events (wind, snow/ice, insects, game, rot, fire) with dates,
regeneration (natural/sowing/planting) and its success, silvicultural measures, non-timber uses (grazing, litter, resin, peat, glassworks),
wildlife, hydrology/land-use changes, and quantities with units (area, volume, growth). Include statements about the past, the present
state and plans/prohibitions, and mark them with "status".

RULES:
1. One finding per distinct statement. Do not merge different stands, dates or species into one finding.
2. "quote" must be copied VERBATIM from the page text (same spelling, including HTR errors), at most ~40 words. Never paraphrase inside quote.
3. Normalise "entity" and species names to modern German headwords (Fichte, Tanne, Buche …), keep "as_written"/"value" as in the text.
4. Convert shares to fractions in share_value (8/10 -> 0.8, 50 % -> 0.5, WZ 0,7 -> 0.7). Do not invent numbers.
5. Administrative content (salaries, accounting, personnel, road costs, signatures) is NOT ecological: skip it, unless it carries
   ecological facts (e.g. wood delivered to a glassworks, grazing rights of villages).
6. Text in blockquotes ("> *[Marginalie]*") are marginal notes: extract from them too, but add "marginalia" in "note".
7. HTML tables inside the text hold stand lists: extract each stand into "locations" (with area) and, if they contain ecological
   attributes (age, species, treatment), also into "findings".
8. The page usually begins and ends mid-sentence. Extract what is on the page; if the subject is unclear because it was on the previous
   page, still extract and set confidence "low" with a note.
9. If the page contains no ecological information, return an empty "findings" list and describe the page in "summary_en".

''' + GLOSSARY + '''
PAGE ID: {page_id}
SOURCE SUBTYPE: {subtype}

=== PAGE TEXT (Markdown) ===
{text}
=== END PAGE TEXT ===
'''

TABLE_PROMPT = '''You are an expert in historical forestry (Forsteinrichtung) and ecological history, reading Gemini HTR transcriptions
of 19th/early-20th-century German forest-management TABLES from the Bavarian Forest (Forstamt St. Oswald, Klingenbrunn, Spiegelau,
Riedlhütte). The table is given as an HTML <table> (with colspan/rowspan, <span class="red"> for red-ink corrections) plus header text
lines above and marginalia below.

TASK: Convert the table into a list of RECORDS – one per stand row (Distrikt / Abteilung / Unterabteilung), sample plot, year row,
or Forstamt total – and extract the ecological content of each record into the JSON structure requested.

RULES:
1. Identify the form first (table_type) from the headers, and list the flattened column headers in "column_headers" so the mapping
   is auditable. Note continuation pages, side-by-side duplicated column blocks (two stands per page) and header/body column drift
   in "parsing_notes".
2. Resolve rowspan/ditto context: a Distrikt or Abteilung given once (rowspan, first row, group header row like "III. Rachelhang")
   applies to the following rows until a new one appears. Ditto marks (", „, ”, ,,, dto, detto, desgleichen) repeat the cell above.
   Cross-references such as "wie bei IV. 8. b" go into "cross_references" – do not resolve them.
3. Numbers split across unit columns (Hektar | Ar; Tagw | Dez; M | Pf) are ONE value: 18 | 60 -> 18.60. Numbers split inside a cell
   ("186 27", "2 03") likewise. Use the decimal point in JSON. Keep the original in the *_raw / raw fields. Do not invent numbers.
4. Free-text description cells ("Beschreibung a) der Lage b) des Bodens c) des Holzbestandes d) der Bewirthschaftung e) Kulturen",
   "Bemerkungen", "Boden / Lage / Holzbestand") are the richest source: copy them verbatim into description_raw and decompose them into
   site (a, b), stand (c), management (d), culture (e), damage and non_timber_use. If a description continues in the next row of the
   same stand, attach it to that stand.
5. Species: normalise names (Fi -> Fichte, Bu/Bü -> Buche, Ta -> Tanne, Lerche -> Lärche …), keep as_written, shares as fractions
   (0,7 -> 0.7). "0,4 bis 0,7 d. Schst" = stocking density 0.4-0.7, not a species share.
6. Rows that are sums, carry-overs (Übertrag, Transport, Summa, Seite 17), or totals per Forstamt get record_kind summary_row /
   carry_over / forest_office_total and only the numeric values that are present.
7. Red-ink content (<span class="red">) is a later correction: put it into red_annotations of the row, and use it for the fields only
   if the black value is missing.
8. Set "uncertain": true for rows with [?], [illegible], or where the alignment of cells to rows is ambiguous.
9. Purely financial columns (Kostenanschlag, Geld) may be captured in culture.cost_* but need no further attention.

''' + GLOSSARY + '''
PAGE ID: {page_id}
TABLE CATEGORY (from folder name): {subtype}

=== PAGE (Markdown with HTML table) ===
{text}
=== END PAGE ===
'''
print("prompts ready:", len(TEXT_PROMPT), "/", len(TABLE_PROMPT), "chars")

## 4 · Gemini call (structured output, retries, cache)

`extract_page()` writes one JSON per page into `output/raw/` so a re-run only processes missing pages.
If the API rejects the JSON schema (schema too complex), the call automatically falls back to
`response_mime_type="application/json"` with the schema embedded in the prompt.

In [ ]:
def _extract_text(resp):
    t = getattr(resp, "text", None)
    if t: return t
    for cand in (getattr(resp, "candidates", None) or []):
        parts = getattr(getattr(cand, "content", None), "parts", None) or []
        got = [p.text for p in parts if getattr(p, "text", None)]
        if got: return "".join(got)
    return ""

def _parse_json_robust(s):
    s = s.strip()
    s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s, flags=re.S)
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        dec = json.JSONDecoder()
        for i, ch in enumerate(s):
            if ch in "{[":
                try:
                    obj, _ = dec.raw_decode(s[i:]); return obj
                except json.JSONDecodeError:
                    continue
    raise ValueError("no JSON object found in response")

def call_gemini(prompt, schema, retries=3, thinking_level=None):
    '''Returns (data, usage_dict, mode). mode = 'schema' | 'json_mime'.'''
    thinking_level = thinking_level or THINKING_LEVEL
    last_err, use_schema = None, True
    for attempt in range(retries + 1):
        cfg = dict(
            max_output_tokens=MAX_OUTPUT_TOKENS,
            thinking_config=types.ThinkingConfig(thinking_level=thinking_level),
            response_mime_type="application/json",
        )
        contents = prompt
        if use_schema:
            cfg["response_json_schema"] = schema
        else:
            contents = prompt + "\n\nReturn ONLY a JSON object that conforms to this JSON schema:\n" + json.dumps(schema, ensure_ascii=False)
        try:
            resp = client.models.generate_content(model=MODEL_ID, contents=contents, config=types.GenerateContentConfig(**cfg))
            um = getattr(resp, "usage_metadata", None)
            usage = {
                "input_tokens": getattr(um, "prompt_token_count", 0) or 0,
                "output_tokens": getattr(um, "candidates_token_count", 0) or 0,
                "thinking_tokens": getattr(um, "thoughts_token_count", 0) or 0,
            }
            txt = _extract_text(resp)
            if not txt:
                raise ValueError("empty response (finish_reason=%s)" % getattr(resp.candidates[0], "finish_reason", "?"))
            return _parse_json_robust(txt), usage, ("schema" if use_schema else "json_mime")
        except Exception as e:
            last_err = e
            msg = str(e).lower()
            if use_schema and ("schema" in msg or "invalid_argument" in msg or "400" in msg):
                use_schema = False          # fall back: schema in prompt
                print("  ! schema rejected, falling back to json_mime:", str(e)[:200])
                continue
            wait = 2.0 * (attempt + 1)
            if "429" in msg or "resource_exhausted" in msg or "quota" in msg:
                wait = 15.0 * (attempt + 1)
            print(f"  ! attempt {attempt+1} failed: {str(e)[:160]} – retry in {wait:.0f}s")
            time.sleep(wait)
    raise RuntimeError(f"Gemini call failed after {retries+1} attempts: {last_err}")

def build_prompt(page):
    tpl = TEXT_PROMPT if page["source_type"] == "text" else TABLE_PROMPT
    return tpl.format(page_id=page["page_id"], subtype=page["subtype"], text=page["text"])

_lock = threading.Lock()
def extract_page(page, force=False):
    out_path = RAW_DIR / f"{page['seq']}.json"
    if out_path.exists() and not force:
        return json.loads(out_path.read_text(encoding="utf-8"))
    schema = TEXT_SCHEMA if page["source_type"] == "text" else TABLE_SCHEMA
    t0 = time.time()
    data, usage, mode = call_gemini(build_prompt(page), schema)
    data.setdefault("page_id", page["page_id"])
    rec = {
        "seq": page["seq"], "source_type": page["source_type"], "subtype": page["subtype"],
        "page_id": page["page_id"], "file": page["file"], "model": MODEL_ID, "thinking_level": THINKING_LEVEL,
        "mode": mode, "usage": usage, "elapsed_s": round(time.time() - t0, 1), "result": data,
    }
    with _lock:
        out_path.write_text(json.dumps(rec, ensure_ascii=False, indent=1), encoding="utf-8")
    return rec

def cost_usd(usage):
    return usage["input_tokens"] / 1e6 * PRICE_INPUT_PER_M + (usage["output_tokens"] + usage["thinking_tokens"]) / 1e6 * PRICE_OUTPUT_PER_M

### 4.1 · Dry run: one text page and one table page

In [ ]:
demo_text  = next(p for p in pages if p["source_type"] == "text")
demo_table = next(p for p in pages if p["source_type"] == "table")

for p in (demo_text, demo_table):
    print("=" * 100); print(p["seq"], p["subtype"], p["page_id"], f"{p['chars']} chars")
    rec = extract_page(p, force=True)
    print(f"mode={rec['mode']}  usage={rec['usage']}  cost≈${cost_usd(rec['usage']):.4f}  {rec['elapsed_s']}s")
    r = rec["result"]
    if p["source_type"] == "text":
        print("summary:", r.get("summary_en"))
        print("species:", [(s.get("name_de"), s.get("role"), s.get("share_value")) for s in r.get("tree_species", [])])
        for f in r.get("findings", [])[:12]:
            print(f"  [{f['category']}/{f.get('subtype','')}] {f['entity']} | {f.get('value','')} {f.get('unit','')} | {f.get('date','')} | {f.get('location','')}\n      « {f['quote']} »")
        print(f"  … {len(r.get('findings', []))} findings total")
    else:
        print("table_type:", r.get("table_type"), "| office:", r.get("forest_office"), "| class:", r.get("operating_class"))
        print("headers:", r.get("column_headers"))
        print("notes:", r.get("parsing_notes"))
        for rc in r.get("records", [])[:6]:
            print(f"  [{rc['record_kind']}] {rc.get('district_no','')} {rc.get('district_name','')} / {rc.get('compartment_no','')} {rc.get('subcompartment','')} | area {rc.get('area_value','')} {rc.get('area_unit','')} | age {rc.get('stand',{}).get('age_raw','')}")
            print("      species:", [(s.get('name_de'), s.get('share_value')) for s in rc.get('stand', {}).get('species', [])])
            print("      site:", {k: v for k, v in rc.get('site', {}).items() if v not in ('', None, [])})
            print("      damage:", rc.get("damage"), "| culture:", rc.get("culture"))
        print(f"  … {len(r.get('records', []))} records total")

### 4.2 · Batch run over the whole test set (parallel, cached)

In [ ]:
FORCE_RERUN = False       # True = ignore cache and re-extract everything
SUBSET = None             # e.g. ["text_01", "table_03"] to run only some pages; None = all

todo = [p for p in pages if (SUBSET is None or p["seq"] in SUBSET)]
results, errors = {}, {}
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(extract_page, p, FORCE_RERUN): p for p in todo}
    for fut in tqdm(as_completed(futs), total=len(futs), desc="extracting"):
        p = futs[fut]
        try:
            results[p["seq"]] = fut.result()
        except Exception as e:
            errors[p["seq"]] = str(e); print("ERROR", p["seq"], str(e)[:200])

results = dict(sorted(results.items()))
tot = {"input_tokens": 0, "output_tokens": 0, "thinking_tokens": 0}
for r in results.values():
    for k in tot: tot[k] += r["usage"].get(k, 0)
print(f"\n{len(results)} pages ok, {len(errors)} errors")
print("tokens:", tot, f"| est. cost ${cost_usd(tot):.3f}  (prices: {PRICE_INPUT_PER_M}/{PRICE_OUTPUT_PER_M} USD per 1M)")
pd.DataFrame([{"seq": r["seq"], "subtype": r["subtype"], "mode": r["mode"], "s": r["elapsed_s"], **r["usage"],
               "n_items": len(r["result"].get("findings", r["result"].get("records", [])))} for r in results.values()])

## 5 · Results: flat tables, quote verification, review viewer

* `text_findings.csv` – one row per finding (all 20 text pages)
* `text_species.csv`, `text_locations.csv`
* `table_records.csv` – one row per stand/plot record with flattened site/stand/management columns
* `table_species.csv`, `table_damage.csv`, `table_culture.csv`, `table_non_timber_use.csv`
* `review_viewer.html` – standalone page: source text with highlighted quotes next to the extracted items

In [ ]:
def _norm(s):
    return re.sub(r"\s+", " ", (s or "")).strip().lower()

def verify_quote(quote, text, min_len=12):
    '''exact -> 'exact'; after whitespace-normalisation -> 'normalised'; first 6 words found -> 'prefix'; else 'missing'.'''
    if not quote: return "empty"
    if quote in text: return "exact"
    nt, nq = _norm(text), _norm(quote)
    if nq in nt: return "normalised"
    words = nq.split()
    if len(words) >= 4 and " ".join(words[:6]) in nt: return "prefix"
    return "missing"

page_by_seq = {p["seq"]: p for p in pages}

# ---------- text pages ----------
find_rows, species_rows, loc_rows, page_rows = [], [], [], []
for seq, r in results.items():
    if r["source_type"] != "text": continue
    res, txt = r["result"], page_by_seq[seq]["text"]
    page_rows.append({"seq": seq, "subtype": r["subtype"], "page_id": r["page_id"], "document_type": res.get("document_type"),
                      "summary_en": res.get("summary_en"), "transcription_quality": res.get("transcription_quality"),
                      "forest_offices": "; ".join(res.get("forest_offices", [])), "dates": "; ".join(res.get("dates_mentioned", [])),
                      "n_findings": len(res.get("findings", []))})
    for i, f in enumerate(res.get("findings", [])):
        find_rows.append({"seq": seq, "source_subtype": r["subtype"], "page_id": r["page_id"], "idx": i,
                          "category": f.get("category"), "finding_subtype": f.get("subtype"),
                          **{k: f.get(k) for k in ("entity", "entity_en", "value", "unit", "date", "location", "status", "confidence", "note")},
                          "species": "; ".join(f.get("species", []) or []), "quote": f.get("quote"),
                          "quote_check": verify_quote(f.get("quote"), txt)})
    for s in res.get("tree_species", []):
        species_rows.append({"seq": seq, "page_id": r["page_id"], **s, "quote_check": verify_quote(s.get("quote"), txt)})
    for l in res.get("locations", []):
        loc_rows.append({"seq": seq, "page_id": r["page_id"], **l, "quote_check": verify_quote(l.get("quote"), txt)})

text_pages_df = pd.DataFrame(page_rows); text_findings = pd.DataFrame(find_rows)
text_species = pd.DataFrame(species_rows); text_locations = pd.DataFrame(loc_rows)

# ---------- tables ----------
rec_rows, tsp_rows, dmg_rows, cul_rows, ntu_rows, hist_rows, tpage_rows = [], [], [], [], [], [], []
for seq, r in results.items():
    if r["source_type"] != "table": continue
    res = r["result"]
    tpage_rows.append({"seq": seq, "subtype": r["subtype"], "page_id": r["page_id"], "table_type": res.get("table_type"),
                       "forest_office": res.get("forest_office"), "operating_class": res.get("operating_class"),
                       "n_records": len(res.get("records", [])), "transcription_quality": res.get("transcription_quality"),
                       "parsing_notes": res.get("parsing_notes"), "column_headers": " | ".join(res.get("column_headers", []) or [])})
    for i, rc in enumerate(res.get("records", [])):
        key = {"seq": seq, "subtype": r["subtype"], "page_id": r["page_id"], "rec": i, "record_kind": rc.get("record_kind"),
               "district_no": rc.get("district_no"), "district_name": rc.get("district_name"),
               "compartment_no": rc.get("compartment_no"), "subcompartment": rc.get("subcompartment")}
        row = dict(key, stand_name=rc.get("stand_name"), forest_office=rc.get("forest_office"), year_or_period=rc.get("year_or_period"),
                   area_value=rc.get("area_value"), area_unit=rc.get("area_unit"), area_raw=rc.get("area_raw"))
        for k, v in (rc.get("site") or {}).items():
            row["site_" + k] = "; ".join(v) if isinstance(v, list) else v
        for k, v in (rc.get("stand") or {}).items():
            if k == "species":
                row["stand_species"] = "; ".join(f"{s.get('name_de')}{'' if s.get('share_value') is None else ' ' + str(s.get('share_value'))}" for s in v)
            else:
                row["stand_" + k] = v
        for k, v in (rc.get("management") or {}).items():
            if k == "history":
                for h in v: hist_rows.append(dict(key, **h))
            else:
                row["mgmt_" + k] = v
        row["n_damage"] = len(rc.get("damage") or []); row["n_culture"] = len(rc.get("culture") or []); row["n_ntu"] = len(rc.get("non_timber_use") or [])
        row["red_annotations"] = "; ".join(rc.get("red_annotations") or []); row["cross_references"] = "; ".join(rc.get("cross_references") or [])
        row["uncertain"] = rc.get("uncertain"); row["description_raw"] = rc.get("description_raw")
        row["description_check"] = verify_quote((rc.get("description_raw") or "")[:120], page_by_seq[seq]["text"]) if rc.get("description_raw") else ""
        rec_rows.append(row)
        for s in rc.get("stand", {}).get("species", []) or []: tsp_rows.append(dict(key, **s))
        for d in rc.get("damage") or []: dmg_rows.append(dict(key, **d))
        for c in rc.get("culture") or []: cul_rows.append(dict(key, **c))
        for n in rc.get("non_timber_use") or []: ntu_rows.append(dict(key, **n))

table_pages_df = pd.DataFrame(tpage_rows); table_records = pd.DataFrame(rec_rows)
table_species = pd.DataFrame(tsp_rows); table_damage = pd.DataFrame(dmg_rows)
table_culture = pd.DataFrame(cul_rows); table_ntu = pd.DataFrame(ntu_rows); table_history = pd.DataFrame(hist_rows)

for name, df in [("text_pages", text_pages_df), ("text_findings", text_findings), ("text_species", text_species), ("text_locations", text_locations),
                 ("table_pages", table_pages_df), ("table_records", table_records), ("table_species", table_species), ("table_damage", table_damage),
                 ("table_culture", table_culture), ("table_non_timber_use", table_ntu), ("table_history", table_history)]:
    df.to_csv(OUT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
    print(f"{name:22s} {len(df):5d} rows")
(OUT_DIR / "all_results.json").write_text(json.dumps(results, ensure_ascii=False, indent=1), encoding="utf-8")

In [ ]:
# ---- overview: findings per category and page, quote verification rate ----
if len(text_findings):
    display(pd.crosstab(text_findings["source_subtype"], text_findings["category"]))
    print("\nquote verification (text findings):"); display(text_findings["quote_check"].value_counts())
    print("\nconfidence:"); display(text_findings["confidence"].value_counts())
    print("\nmost frequent entities:"); display(text_findings["entity"].value_counts().head(25))
if len(table_records):
    print("\nrecords per table page / kind:"); display(pd.crosstab(table_records["subtype"], table_records["record_kind"]))
    print("\ndescription_raw verification:"); display(table_records["description_check"].value_counts())
    print("\nspecies in tables:"); display(table_species["name_de"].value_counts().head(15))

In [ ]:
# ---- inspect: all findings of one text page / all records of one table page ----
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 250)
SEQ = "text_09"      # storm 1870 / Borkenkäfer 1873/74 / Weide / Harz page
display(text_findings[text_findings.seq == SEQ][["category", "finding_subtype", "entity", "value", "unit", "date", "location", "status", "confidence", "quote_check", "quote"]])
SEQ = "table_02"
display(table_records[table_records.seq == SEQ][[c for c in table_records.columns if c.startswith(("district", "compartment", "subcomp", "area_value", "site_", "stand_species", "stand_age", "n_", "uncertain"))]])

### 5.1 · Review viewer (standalone HTML, source text with highlighted quotes)

In [ ]:
COLORS = {"tree_species": "#c8e6c9", "stand_structure": "#dcedc8", "site_conditions": "#ffe0b2", "ground_vegetation": "#f0f4c3",
          "climate_weather": "#b3e5fc", "damage_event": "#ffcdd2", "regeneration": "#d1c4e9", "silvicultural_measure": "#e1bee7",
          "non_timber_use": "#ffecb3", "wildlife": "#f8bbd0", "land_use_hydrology": "#b2dfdb", "quantity": "#e0e0e0", "other_ecological": "#eeeeee"}

def highlight(text, items):
    '''items: list of (quote, category, idx). Greedy longest-first, non-overlapping.'''
    spans = []
    for q, cat, idx in sorted(items, key=lambda x: -len(x[0] or "")):
        if not q: continue
        start = text.find(q)
        if start < 0:  # try normalised match on first words
            m = re.search(re.escape(" ".join(q.split()[:5])).replace(r"\ ", r"\s+"), text)
            if not m: continue
            start, end = m.start(), min(len(text), m.start() + len(q))
        else:
            end = start + len(q)
        if any(not (end <= s or start >= e) for s, e, *_ in spans): continue
        spans.append((start, end, cat, idx))
    spans.sort()
    out, pos = [], 0
    for s, e, cat, idx in spans:
        out.append(html.escape(text[pos:s]))
        out.append(f'<mark style="background:{COLORS.get(cat, "#eee")}" title="{html.escape(str(cat))} #{idx}">{html.escape(text[s:e])}<sup>{idx}</sup></mark>')
        pos = e
    out.append(html.escape(text[pos:]))
    return "".join(out)

def _species_str(rc):
    sp = (rc.get("stand") or {}).get("species", []) or []
    return ", ".join(f"{s.get('name_de')} {s.get('share_value') if s.get('share_value') is not None else '?'}" for s in sp)

def page_html(seq):
    p, r = page_by_seq[seq], results[seq]; res = r["result"]
    if p["source_type"] == "text":
        items = [(f.get("quote"), f.get("category"), i) for i, f in enumerate(res.get("findings", []))]
        left = f'<pre class="src">{highlight(p["text"], items)}</pre>'
        rows = "".join(f'<tr style="background:{COLORS.get(f.get("category"), "#eee")}"><td>{i}</td><td>{html.escape(str(f.get("category")))}<br><small>{html.escape(str(f.get("subtype") or ""))}</small></td>'
                       f'<td><b>{html.escape(str(f.get("entity")))}</b><br><small>{html.escape(str(f.get("entity_en") or ""))}</small></td>'
                       f'<td>{html.escape(str(f.get("value") or ""))} {html.escape(str(f.get("unit") or ""))}</td><td>{html.escape(str(f.get("date") or ""))}</td>'
                       f'<td>{html.escape(str(f.get("location") or ""))}</td><td>{html.escape(str(f.get("status") or ""))}</td><td>{html.escape(str(f.get("confidence")))}</td>'
                       f'<td><small>{html.escape(str(f.get("note") or ""))}</small></td></tr>' for i, f in enumerate(res.get("findings", [])))
        sp = ", ".join(f'{s.get("name_de")} ({s.get("role")}{"" if s.get("share_value") is None else ", " + str(s.get("share_value"))})' for s in res.get("tree_species", []))
        right = (f'<p><b>{html.escape(str(res.get("document_type")))}</b> · quality {res.get("transcription_quality")}<br>{html.escape(str(res.get("summary_en")))}</p>'
                 f'<p><b>Species:</b> {html.escape(sp)}</p>'
                 f'<table><tr><th>#</th><th>category</th><th>entity</th><th>value</th><th>date</th><th>location</th><th>status</th><th>conf</th><th>note</th></tr>{rows}</table>')
    else:
        items = [((rc.get("description_raw") or "")[:200], "site_conditions", i) for i, rc in enumerate(res.get("records", []))]
        left = f'<div class="src">{p["text"].replace("```html", "").replace("```", "")}</div>' if "<table" in p["text"] else f'<pre class="src">{html.escape(p["text"])}</pre>'
        right = (f'<p><b>{html.escape(str(res.get("table_type")))}</b> · {html.escape(str(res.get("forest_office") or ""))} · {html.escape(str(res.get("operating_class") or ""))}'
                 f'<br><small>headers: {html.escape(" | ".join(res.get("column_headers", []) or []))}</small><br><small>notes: {html.escape(str(res.get("parsing_notes") or ""))}</small></p>'
                 + "".join(f'<details {"open" if i < 3 else ""}><summary><b>#{i} {rc.get("record_kind")}</b> {rc.get("district_no") or ""} {rc.get("district_name") or ""} / {rc.get("compartment_no") or ""} {rc.get("subcompartment") or ""} '
                           f'· {rc.get("area_value") or ""} {rc.get("area_unit") or ""} · species {html.escape(_species_str(rc))}</summary>'
                           f'<pre>{html.escape(json.dumps({k: v for k, v in rc.items() if v not in ("", None, [], {})}, ensure_ascii=False, indent=1))}</pre></details>'
                           for i, rc in enumerate(res.get("records", []))))
    return f'<section id="{seq}" class="page"><h2>{seq} · {html.escape(p["subtype"])} · {html.escape(p["page_id"])}</h2><div class="cols"><div class="left">{left}</div><div class="right">{right}</div></div></section>'

nav = "".join(f'<option value="{seq}">{seq} · {page_by_seq[seq]["subtype"]} · {page_by_seq[seq]["page_id"]}</option>' for seq in results)
legend = " ".join(f'<span style="background:{c};padding:2px 6px;border-radius:3px">{k}</span>' for k, c in COLORS.items())
viewer = f'''<!doctype html><html><head><meta charset="utf-8"><title>Forsteinrichtung IE review</title>
<style>body{{font-family:system-ui,sans-serif;margin:0;padding:12px}} .cols{{display:flex;gap:12px}} .left,.right{{flex:1;min-width:0;overflow:auto;max-height:85vh;border:1px solid #ccc;padding:8px}}
.src{{white-space:pre-wrap;font-size:13px;line-height:1.45}} table{{border-collapse:collapse;font-size:12px}} td,th{{border:1px solid #bbb;padding:3px 5px;vertical-align:top}}
mark sup{{font-size:9px}} .page{{display:none}} .page.active{{display:block}} pre{{font-size:11px;white-space:pre-wrap}} .right table td{{font-size:11px}}
.src table{{font-size:11px}} .src td,.src th{{border:1px solid #999;padding:2px 4px}} .src .red{{color:#c00}}</style></head><body>
<h1>Forsteinrichtung – ecological IE review ({MODEL_ID}, thinking={THINKING_LEVEL})</h1>
<p>{legend}</p><select id="sel" onchange="show(this.value)" style="width:100%;font-size:14px">{nav}</select>
{"".join(page_html(seq) for seq in results)}
<script>function show(id){{document.querySelectorAll('.page').forEach(e=>e.classList.toggle('active',e.id===id));}} show(document.getElementById('sel').value);</script>
</body></html>'''
(OUT_DIR / "review_viewer.html").write_text(viewer, encoding="utf-8")
print("written:", OUT_DIR / "review_viewer.html", f"({len(viewer)//1024} kB)")
from IPython.display import HTML, display
display(HTML(f'<iframe srcdoc="{html.escape(viewer)}" style="width:100%;height:700px;border:1px solid #888"></iframe>'))

### 5.2 · Manual scoring sheet and download

In [ ]:
# scoring template: one row per finding / record; fill 'verdict' with correct | partial | wrong | missing-context
score = []
if len(text_findings):
    score.append(text_findings[["seq", "page_id", "idx", "category", "entity", "value", "date", "location", "confidence", "quote_check", "quote"]].assign(kind="text_finding", verdict="", comment=""))
if len(table_records):
    score.append(table_records[["seq", "page_id", "rec", "record_kind", "district_no", "compartment_no", "subcompartment", "area_value", "stand_species", "stand_age_raw", "uncertain", "description_check"]]
                 .rename(columns={"rec": "idx"}).assign(kind="table_record", verdict="", comment=""))
scoring = pd.concat(score, ignore_index=True)
scoring.to_csv(OUT_DIR / "scoring_sheet.csv", index=False, encoding="utf-8-sig")
print("scoring sheet rows:", len(scoring))

zip_path = shutil.make_archive(str(WORK / f"Forsteinrichtung_IE_results_{time.strftime('%Y%m%d_%H%M')}"), "zip", OUT_DIR)
print("results zip:", zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

## 6 · Notes for scaling to the full corpus

**What the two surveys of the combined folder showed**

* Text pages: 1140 md files (Forsteinrichtung 623, Schreiben 216, Protokolle 184, Gedruckter Text 117); median 1.2–2.4 k chars.
  Nearly every page starts/ends mid-sentence → for the full run, pass the tail of the previous page (same signature, consecutive `[NNN]`)
  as context, or extract per document sequence and merge. Skip pages < 300 body chars. Duplicated fair copies exist
  (388 IV ≈ 388 II; 389 II ≈ 389 III; several pages twice within 388 IV) – deduplicate before aggregating counts.
  One page (`[033] … A 379 II`) has a 165 kB marginalia loop → the truncation above handles it.
* Tables: 2943 md files. Ecologically rich forms: **Periodentabelle** (367 substantive pages with the a)–e) description column),
  Wirtschaftsplan (231), Probeflächenaufnahmen (20), Kulturplan (103), Bestandsauszählungen (315), Altersklassen (117),
  Streunutzungsplan (9), Forstnebennutzungen (13), Übersicht der Forstverbesserungen (10), Wirtschaftsbuch (349, harvest chronology only).
  Skip (financial/computational): Kostenanschläge, Tarif, Kreisflächen, Kubirung, Walzen-Stücke, Nivellements, Wegbauplan,
  Triftbauplan, Controle-Hauptbuch, Taxations-Soll/Haben, Abschluss der Wirtschaftsbücher, Fällungsnachweisungen, Änderungen am Detail.
  Median table page 7.4 k chars; 90 % ≤ 15 k; a few Betriebsplan/Entwurf pages reach 30–59 k (still far below the 1M context).
* Stand keys need normalising across pages (`Reg NB` vs `RegNB`, `KdForsten` vs `KdForten`, `[012|` filenames, Roman numerals,
  `lit.`/`Litt.`), and cross-references like "wie bei IV. 8. b" can only be resolved after all pages of a volume are extracted.

The cell below enumerates the full corpus with the priority list, so the same `extract_page()` can be run on it
(only `manifest`/`pages` change). Cost scales with the token counts printed in section 4.2.

In [ ]:
COMBINED_ROOT = Path("/content/drive/MyDrive/Forsteinrichtungsoperate_combined") if IN_COLAB else Path("/home/tobiasperschl/Downloads/Forsteinrichtungsoperate_combined")

TABLE_PRIORITY = ["Periodentabelle", "Wirtschaftsplan", "Probeflaechenaufnahmen", "Kulturplan", "Bestandsauszaehlungen", "Altersklassen",
                  "Streunutzungsplan", "Streunutzungsplan-Weideschaft_und_Streunutzung", "Forstnebennutzungen", "Uebersicht-der-Forstverbesserungen",
                  "Materialvorraete", "Betriebsplan", "Entwurf-des-generellen-Betriebsplans", "Angriffshiebe", "Vermessungstabelle", "Wirtschaftsbuch", "nicht-gegeordnet"]
TABLE_SKIP = {"Kostenanschlaege", "Tarif-für-die-Holzpreise", "Kreisflaechen-fuer-gegebene-Durchmesser", "Kubirung", "Berechnung-von-Walzen-Stuecken",
              "Nivellements_Tabellen", "Wegbauplan", "Spezieller_Triftbauplan", "Controle_Hauptbuch", "Zusammenstellung_des_Taxations_Solls_und_Habens",
              "Abschluss-der-Wirtschaftsbuecher", "Faellungsnachweisungen", "Aenderungen_am-staendigen_und_unstaendigen_Detail", "Gliederungen",
              "Materialertrag-Scans-in-schlecher-Qualität", "Materialergebnisse", "Massenberechnungen", "Materialberechnungen",
              "Streunutzungsplan-Uebersicht_der_an_den_Kartensteinen_vorgenommenen_Aenderungen", "Zusammenstellung-der-Inproduktiven-Flächen", "Verschiedenes"}

def enumerate_corpus(root, min_chars=300):
    rows = []
    for p in sorted((root / "Textseiten").rglob("md/*.md")):
        sub = p.relative_to(root / "Textseiten").parts[0].replace("Textseiten-", "")
        if p.stat().st_size >= min_chars:
            rows.append({"seq": f"text_{len(rows):05d}", "source_type": "text", "subtype": sub, "page_id": p.stem, "file": str(p), "chars": p.stat().st_size})
    for p in sorted((root / "Tabellen").rglob("md/*.md")):
        cat = p.relative_to(root / "Tabellen").parts[0]
        if cat in TABLE_SKIP or p.stat().st_size < min_chars: continue
        rows.append({"seq": f"table_{len(rows):05d}", "source_type": "table", "subtype": cat, "page_id": p.stem, "file": str(p), "chars": p.stat().st_size,
                     "priority": TABLE_PRIORITY.index(cat) if cat in TABLE_PRIORITY else 99})
    return pd.DataFrame(rows)

if COMBINED_ROOT.exists():
    corpus = enumerate_corpus(COMBINED_ROOT)
    print(corpus.groupby(["source_type", "subtype"]).agg(pages=("file", "count"), chars=("chars", "sum")).sort_values(["source_type", "pages"], ascending=[True, False]))
    est_in_tokens = corpus["chars"].sum() / 3.2 + len(corpus) * 1800   # ~3.2 chars/token for German + prompt overhead
    print(f"\n{len(corpus)} pages · ≈{est_in_tokens/1e6:.1f} M input tokens · ≈${est_in_tokens/1e6*PRICE_INPUT_PER_M:.0f} input cost (output/thinking extra)")
else:
    print("combined corpus not found at", COMBINED_ROOT, "– copy it to Drive (or adjust COMBINED_ROOT) for the full run")